In [31]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# 1. Economic & Route Constants
BRIDGE_TARGET = 'WYALUSING RIVER BRIDGE'
DETOUR_TIME_MINS = 25
DETOUR_TIME_HOURS = DETOUR_TIME_MINS / 60
DETOUR_DISTANCE_MILES = 12.5

# Value of Time (VOT) per hour
PASSENGER_TIME_VALUE = 25.00
COMMERCIAL_TIME_VALUE = 75.00

# Vehicle Efficiency (MPG)
PASSENGER_MPG = 26.5
COMMERCIAL_MPG = 6.5

# Local Fuel Prices (April 2026)
AVG_GAS_PRICE = 4.19
AVG_DIESEL_PRICE = 5.99

In [9]:
bridge_data = pd.read_csv('../data/Pennsylvania_Bridges.csv')

C:\Users\rharv\AppData\Local\Temp\ipykernel_17768\3330314245.py:1: DtypeWarning: Columns (0: APPR_DKMEMBTYPE, 1: APPR_DKPROTECT, 2: APPR_DKSURFTYPE, 3: BB_BRDGEID, 4: BRIDGE_ID, 5: CONGRESS_DISTRICTS, 6: CUSTODIAN, 7: FUNCCLASS, 8: HISTSIGN, 9: MATERIALMAIN, 10: NAVCNTROL, 11: SEG_NO, 12: SENATE_DISTRICTS) have mixed types. Specify dtype option on import or set low_memory=False.
  bridge_data = pd.read_csv('../data/Pennsylvania_Bridges.csv')


In [18]:
useful_columns = [
    'BRKEY',                     # Primary Key
    'FACILITY',                  # Road Name (SR 2010)
    'LOCATION',                  # Description
    'YEARBUILT',                 # Age
    'ADTTOTAL',                  # Total Average Daily Traffic (AADT)
    'TRUCKPCT',                  # % of Trucks
    'VCLROVER',                  # Vertical Clearance Over
    'MIN_OVER_VERT_CLEAR_LEFT',  # Clearance (Left)
    'MIN_OVER_VERT_CLEAR_RIGHT', # Clearance (Right)
    'SUFF_RATE',                 # Sufficiency Rating
    'DKRATING',                  # Deck Condition
    'SUPRATING',                 # Superstructure Condition
    'SUBRATING',                 # Substructure Condition
    'CONDITION'                  # Overall Status
]

In [24]:
bridge_data_cleaned = bridge_data[useful_columns].copy()

In [35]:
rainbow_bridge = bridge_data_cleaned[bridge_data_cleaned['LOCATION'] == 'WYALUSING RIVER BRIDGE']

In [37]:
rainbow_bridge.head()

,BRKEY,FACILITY,LOCATION,YEARBUILT,ADTTOTAL,TRUCKPCT,VCLROVER,MIN_OVER_VERT_CLEAR_LEFT,MIN_OVER_VERT_CLEAR_RIGHT,SUFF_RATE,DKRATING,SUPRATING,SUBRATING,CONDITION
50695,6370,SR 2010,WYALUSING RIVER BRIDGE,1942.0,5562.0,13.0,14.83,99.9,14.83,50.4,6,5,5,F


In [47]:
aadt = rainbow_bridge['ADTTOTAL'].iloc[0]
truck_pct = rainbow_bridge['TRUCKPCT'].iloc[0]

# 2. Convert percentage to a raw count (The Fix)
adtt = (aadt * truck_pct) / 100
passenger_count = aadt - adtt

In [52]:
# TIME IMPACT (Opportunity Cost)
daily_passenger_time_cost = passenger_count * DETOUR_TIME_HOURS * PASSENGER_TIME_VALUE
daily_commercial_time_cost = adtt * DETOUR_TIME_HOURS * COMMERCIAL_TIME_VALUE
total_daily_time_cost = daily_passenger_time_cost + daily_commercial_time_cost

In [53]:
daily_passenger_fuel_cost = passenger_count * (DETOUR_DISTANCE_MILES / PASSENGER_MPG) * AVG_GAS_PRICE
daily_commercial_fuel_cost = adtt * (DETOUR_DISTANCE_MILES / COMMERCIAL_MPG) * AVG_DIESEL_PRICE
total_daily_fuel_cost = daily_passenger_fuel_cost + daily_commercial_fuel_cost

In [54]:
total_daily_impact = total_daily_time_cost + total_daily_fuel_cost

days_in_month = 30.42
monthly_impact = total_daily_impact * days_in_month
total_impact_4_months = total_daily_impact * 122
total_impact_6_months = total_daily_impact * 183
annual_impact = total_daily_impact * 365

In [55]:
# 5. Results Summary
print(f"--- ECONOMIC IMPACT SUMMARY: {BRIDGE_TARGET} ---")
print(f"Total Daily Time Loss:   ${total_daily_time_cost:,.2f}")
print(f"Total Daily Fuel Cost:   ${total_daily_fuel_cost:,.2f}")
print("-" * 45)
print(f"TOTAL DAILY ECONOMIC IMPACT: ${total_daily_impact:,.2f}")
print(f"Projected Monthly Impact:    ${monthly_impact:,.2f}")
print(f"Projected 4-Month Burden:    ${total_impact_4_months:,.2f}")
print(f"Projected 6-Month Burden:    ${total_impact_6_months:,.2f}")
print(f"TOTAL ANNUALIZED IMPACT:     ${annual_impact:,.2f}")

--- ECONOMIC IMPACT SUMMARY: WYALUSING RIVER BRIDGE ---
Total Daily Time Loss:   $73,001.25
Total Daily Fuel Cost:   $17,892.85
---------------------------------------------
TOTAL DAILY ECONOMIC IMPACT: $90,894.10
Projected Monthly Impact:    $2,764,998.49
Projected 4-Month Burden:    $11,089,080.08
Projected 6-Month Burden:    $16,633,620.13
TOTAL ANNUALIZED IMPACT:     $33,176,346.16
